# Neural Networks / Multi-Layer Perceptron

---

## Overview

A **Dense Neural Network** (Multi-Layer Perceptron) stacks multiple layers of neurons. Each layer applies an affine transformation followed by a nonlinear activation:

$$\mathbf{z}^{[l]} = W^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$
$$\mathbf{a}^{[l]} = \sigma\left(\mathbf{z}^{[l]}\right)$$

where $\sigma(z) = \dfrac{1}{1 + e^{-z}}$ is the sigmoid activation.

---

## Backpropagation

The output layer error delta:
$$\boldsymbol{\delta}^{[L]} = (\mathbf{a}^{[L]} - \mathbf{y}) \odot \sigma'(\mathbf{z}^{[L]})$$

Hidden layer deltas (chain rule):
$$\boldsymbol{\delta}^{[l]} = \left(W^{[l+1]\top} \boldsymbol{\delta}^{[l+1]}\right) \odot \sigma'(\mathbf{z}^{[l]})$$

Weight updates:
$$W^{[l]} \leftarrow W^{[l]} - \alpha \, \boldsymbol{\delta}^{[l]} \mathbf{a}^{[l-1]\top}$$

---

**Dataset:** MNIST digits (sklearn built-in. 8×8 pixel images)  
**Task:** Multi-class digit classification (0–9).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme()

from rice_ml.supervised_learning.multilayer_perceptron import DenseNetwork
from rice_ml.preprocess import StandardScaler, train_test_split
from rice_ml.metrics import accuracy_score, confusion_matrix

In [ ]:
try:
    import pandas as pd
    df = pd.read_csv('../../../data/Obesity_levels.csv')
    from rice_ml.preprocess import OrdinalEncoder
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        enc = OrdinalEncoder()
        df[col] = enc.fit_transform(df[[col]]).ravel()
    target_col = 'NObeyesdad' if 'NObeyesdad' in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col]).values.astype(float)
    y_raw = df[target_col].values
    classes, y = np.unique(y_raw, return_inverse=True)
    n_classes = len(classes)
    print(f'Loaded obesity dataset: {X.shape}, {n_classes} classes')
except (FileNotFoundError, Exception):
    from sklearn.datasets import load_digits
    data = load_digits()
    X, y = data.data, data.target
    n_classes = 10
    print(f'CSV not found. using digits dataset: {X.shape}')

print(f'Input features: {X.shape[1]}, Classes: {n_classes}')

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## Build and Train the DenseNetwork

Architecture: `[n_features, 64, 32, n_classes]`  
Weights are initialized with **He initialization**: scale = $\sqrt{2 / n_{\text{in}}}$

In [ ]:
n_features = X_train.shape[1]
net = DenseNetwork(layers=[n_features, 64, 32, n_classes])
net.fit(X_train, y_train, alpha=0.01, epochs=100)

plt.figure(figsize=(10, 6))
plt.plot(net.errors_, color='steelblue')
plt.xlabel('Epoch', fontsize=15)
plt.ylabel('MSE Cost', fontsize=15)
plt.title('DenseNetwork: Training Cost per Epoch', fontsize=18)
plt.show()

In [ ]:
# Train/test accuracy at epoch checkpoints. diagnoses overfitting
checkpoints = list(range(10, 101, 10))
train_accs, test_accs = [], []

for ep in checkpoints:
    net_cp = DenseNetwork(layers=[n_features, 64, 32, n_classes])
    net_cp.fit(X_train, y_train, alpha=0.01, epochs=ep)
    train_accs.append(accuracy_score(y_train, net_cp.predict(X_train)))
    test_accs.append(accuracy_score(y_test, net_cp.predict(X_test)))

plt.figure(figsize=(10, 6))
plt.plot(checkpoints, train_accs, marker='o', label='Train Accuracy', color='steelblue')
plt.plot(checkpoints, test_accs, marker='s', label='Test Accuracy', color='salmon')
plt.xlabel('Epochs', fontsize=15)
plt.ylabel('Accuracy', fontsize=15)
plt.title('DenseNetwork: Train vs Test Accuracy', fontsize=18)
plt.legend(fontsize=13)
plt.tight_layout()
plt.show()

print(f'Train accuracy at 100 epochs: {train_accs[-1]:.4f}')
print(f'Test  accuracy at 100 epochs: {test_accs[-1]:.4f}')
gap = train_accs[-1] - test_accs[-1]
print(f'Generalization gap:           {gap:.4f}  (>0.05 may indicate overfitting)')


In [ ]:
y_pred = net.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'Test Accuracy: {acc:.4f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted', fontsize=13)
plt.ylabel('True', fontsize=13)
plt.title('DenseNetwork: Confusion Matrix', fontsize=16)
plt.show()

## Interpretation

- The cost decreasing each epoch confirms that backpropagation is correctly propagating gradients.
- Diagonal entries in the confusion matrix represent correct predictions per class.
- Deeper networks and more epochs generally improve accuracy, at the cost of training time.